In [1]:
print("Hola, este es mi primer chatbot")

Hola, este es mi primer chatbot


In [2]:
%pip -q install nltk
import sys
import nltk
print("Python:", sys.version.split()[0])
print("NLTK:", nltk.__version__)
assert sys.version_info >= (3, 10), "Usa Python 3.10 o superior"


Python: 3.13.16
NLTK: 3.9.1


In [3]:
import unicodedata
from nltk.tokenize import RegexpTokenizer
tokenizador = RegexpTokenizer(r"\w+")
def normalizar(texto):
 texto = texto.lower()
 texto = unicodedata.normalize("NFD", texto)
 texto = "".join(
 letra for letra in texto
 if unicodedata.category(letra) != "Mn"
 )
 palabras = tokenizador.tokenize(texto)
 return " ".join(palabras)
print(normalizar("¿DÓNDE está el MENÚ?"))

donde esta el menu


In [56]:
intenciones = {
 "saludo": {
 "reglas": ["hola", "buenos dias", "buenas tardes","Que tal"],
 "respuesta": "Hola. puede consultar su fecha entrega, el estatus de entrega, aprobacion, cola o turno,etc."
 },
 "estatus": {
 "reglas": ["estatus", "cual es el estatus"],
 "respuesta": "Su orden se encuentra en proceso de fabricacion."
 },
 "entrega": {
 "reglas": ["entrega", "cuando se entrega","fecha de entrega"],
 "respuesta": "Su orden esta programada para entregarse el 20 de noviebre."
 },
 "aprobacion": {
 "reglas": ["aprobacion", "ya esta aprobada"],
 "respuesta": "Su orden esta completamente aprobada."
 }
}
print("Intenciones cargadas:", len(intenciones))
interacciones = 1

Intenciones cargadas: 4


In [57]:
intenciones.update({
 "cola": {
 "reglas": ["delante", "antes que el mio","que lugar tiene","turno"],
 "respuesta": "Actualmente su trabajo tiene otros 4 pedidos por delante."
 },
 "adelantar": {
 "reglas": ["adelantar", "entregar antes"],
 "respuesta": "Podemos adelantar sus trabajo si se autoriza trabajar tiempo extra."
 },
 "terminado": {
 "reglas": ["terminado", "completo","puedo pasar por el"],
 "respuesta": "No, su trabajo esta en proceso de fabricacion."
 },
 "sabado": {
 "reglas": ["sabado", "trabajar el sabado"],
 "respuesta": "No, actualmente no tenemos autorizado el tiempo extra"
 },
 "cierre": {
 "reglas": ["adios", "hasta luego", "gracias","terminar sesion","cerrar","salir","cierre"],
 "respuesta": "Nos vemos pronto, sigo pendiente a sus preguntas."
 },
 "prohibido": {
 "reglas": ["trabajos de", "pendiente de"],
 "respuesta": "No estoy autorizado para responder a su pregunta."
 },
 "costo": {
 "reglas": ["costo", "precio","Cuanto va a costar"],
 "respuesta": "El costo de su trabajo es de 150 usd."
 }
})
print("Intenciones cargadas:", len(intenciones))

Intenciones cargadas: 11


In [39]:
def responder(mensaje):
 texto = normalizar(mensaje)
 coincidencias = []
 for nombre, datos in intenciones.items():
  for regla in datos["reglas"]:
    patron = " " + normalizar(regla) + " "
    if patron in " " + texto + " ":
      coincidencias.append(nombre)
      break
 if len(coincidencias) == 1:
  nombre = coincidencias[0]
  return nombre, intenciones[nombre]["respuesta"]
 if len(coincidencias) > 1:
  return "ambigua", "Consulta un solo tema por mensaje."
 return "desconocida", "No tengo datos para darle una respuesta, pruebe preguntando otra cosa"
print(responder("¿Cuál el el estatus de mi orden?"))


('estatus', 'Su orden se encuentra en proceso de fabricacion.')


In [70]:
print("Tool Room  - Chatbot para seguimiento a su orden de trabajo. Interaccion #" + str(interacciones))
while True:
 mensaje = input("Tu: ")
 intencion, respuesta = responder(mensaje)
 print("Intencion:", intencion)
 print("Bot:", respuesta)
 if intencion == "salir" or intencion =="cierre":
  interacciones = interacciones +1
  break

Tool Room  - Chatbot para seguimiento a su orden de trabajo. Interaccion #12
Tu: Hola, cual es el estatus de mi orden?
Intencion: ambigua
Bot: Consulta un solo tema por mensaje.
Tu: hola
Intencion: saludo
Bot: Hola. puede consultar su fecha entrega, el estatus de entrega, aprobacion, cola o turno,etc.
Tu: cual es estatus de mi orden ?
Intencion: estatus
Bot: Su orden se encuentra en proceso de fabricacion.
Tu: gracias
Intencion: cierre
Bot: Nos vemos pronto, sigo pendiente a sus preguntas.


In [9]:
pruebas = [
 ("Hola", "saludo"),
 ("¿Cuál es el HORARIO?", "horario"),
 ("¿Dónde están?", "ubicacion"),
 ("¿Qué venden?", "menu"),
 ("¿Cuánto cuesta?", "precios"),
 ("¿Aceptan tarjeta?", "pagos"),
 ("Quiero pedir", "pedido"),
 ("Adiós", "despedida"),
 ("¿Venden computadoras?", "desconocida"),
 ("horario y menu", "ambigua"),
 ("", "desconocida"),
 ("salir", "salir")
]
aciertos = 0
for mensaje, esperada in pruebas:
 detectada, respuesta = responder(mensaje)
 correcto = detectada == esperada
 aciertos += int(correcto)
 print("Entrada:", repr(mensaje))
 print("Esperada:", esperada, "Detectada:", detectada)
 print("Bot:", respuesta, "| Cumple:", correcto)
print("Coincidencias:", aciertos, "/", len(pruebas))

Entrada: 'Hola'
Esperada: saludo Detectada: saludo
Bot: Hola. Consulta horario, menu o pagos. | Cumple: True
Entrada: '¿Cuál es el HORARIO?'
Esperada: horario Detectada: horario
Bot: Abrimos de lunes a sabado, de 8 a 20 h. | Cumple: True
Entrada: '¿Dónde están?'
Esperada: ubicacion Detectada: ubicacion
Bot: Estamos en Calle Ejemplo 123, Irapuato. | Cumple: True
Entrada: '¿Qué venden?'
Esperada: menu Detectada: menu
Bot: Ofrecemos cafe americano, te y sandwich. | Cumple: True
Entrada: '¿Cuánto cuesta?'
Esperada: precios Detectada: precios
Bot: Americano: $35; te: $30; sandwich: $60. | Cumple: True
Entrada: '¿Aceptan tarjeta?'
Esperada: pagos Detectada: pagos
Bot: Aceptamos efectivo y tarjeta. | Cumple: True
Entrada: 'Quiero pedir'
Esperada: pedido Detectada: pedido
Bot: Solicita tu pedido al personal en caja. | Cumple: True
Entrada: 'Adiós'
Esperada: despedida Detectada: despedida
Bot: Gracias por consultar. Hasta pronto. | Cumple: True
Entrada: '¿Venden computadoras?'
Esperada: descono